# 06 — Train the waveform ResNet1D

Training uses only train subjects; early stopping and model selection use
validation subjects. The held-out test split is deliberately not evaluated here.
Class weighting addresses the 830/3,058 good/poor imbalance without duplicating
subjects or manufacturing new labels.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
WAVEFORM_DIR = Path(
    os.environ.get("PURE_WAVEFORM_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-waveforms-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Waveform dataset:", WAVEFORM_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset

from app.models.waveform_resnet import PPGQualityResNet1D, parameter_count

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
archive = np.load(WAVEFORM_DIR / "waveforms.npz")
X = archive["X"].astype(np.float32)
y = archive["y"].astype(np.float32)
split = archive["split"].astype(str)

train_indices = np.flatnonzero(split == "train")
validation_indices = np.flatnonzero(split == "validation")

class PPGDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        index = self.indices[item]
        return torch.from_numpy(X[index]), torch.tensor(y[index], dtype=torch.float32)

train_loader = DataLoader(PPGDataset(train_indices), batch_size=64, shuffle=True, num_workers=0)
validation_loader = DataLoader(PPGDataset(validation_indices), batch_size=128, shuffle=False, num_workers=0)
print("Train / validation:", len(train_indices), len(validation_indices))

In [ ]:
def predict(model, loader):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for inputs, targets in loader:
            logits.append(model(inputs.to(DEVICE)).cpu().numpy())
            labels.append(targets.numpy())
    logits = np.concatenate(logits)
    labels = np.concatenate(labels).astype(int)
    probabilities = 1.0 / (1.0 + np.exp(-np.clip(logits, -30, 30)))
    return labels, logits, probabilities


def classification_metrics(labels, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "accuracy": float(np.mean(predictions == labels)),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision_good": precision_score(labels, predictions, zero_division=0),
        "recall_good": recall_score(labels, predictions, zero_division=0),
        "f1_good": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probabilities),
        "average_precision": average_precision_score(labels, probabilities),
    }

In [ ]:
model = PPGQualityResNet1D().to(DEVICE)
negative_count = float(np.sum(y[train_indices] == 0))
positive_count = float(np.sum(y[train_indices] == 1))
positive_weight = torch.tensor([negative_count / positive_count], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-5
)

print("Trainable parameters:", parameter_count(model))
print("Positive-class loss weight:", float(positive_weight.item()))

In [ ]:
MAX_EPOCHS = 60
EARLY_STOPPING_PATIENCE = 10
checkpoint_directory = MODEL_DIR / "candidates"
checkpoint_directory.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_directory / "waveform_resnet1d.pt"

history = []
best_score = -np.inf
epochs_without_improvement = 0
started = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    batch_losses = []
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(inputs), targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        batch_losses.append(float(loss.item()))

    labels_val, _, probability_val = predict(model, validation_loader)
    metrics_val = classification_metrics(labels_val, probability_val)
    score = metrics_val["balanced_accuracy"]
    scheduler.step(score)
    history.append({
        "epoch": epoch,
        "train_loss": float(np.mean(batch_losses)),
        "learning_rate": optimizer.param_groups[0]["lr"],
        **metrics_val,
    })
    print(
        f"epoch={epoch:02d} loss={history[-1]['train_loss']:.4f} "
        f"val_bal_acc={score:.4f} val_auc={metrics_val['roc_auc']:.4f} "
        f"val_f1={metrics_val['f1_good']:.4f}"
    )

    if score > best_score + 1e-4:
        best_score = score
        epochs_without_improvement = 0
        torch.save(model.state_dict(), checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping")
            break

elapsed_seconds = time.time() - started
print(f"Best validation balanced accuracy: {best_score:.4f}")
print(f"Training time: {elapsed_seconds:.1f}s")

In [ ]:
import pandas as pd

history_frame = pd.DataFrame(history)
display(history_frame.tail())
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
history_frame.plot(x="epoch", y="train_loss", ax=axes[0], title="Training loss")
history_frame.plot(x="epoch", y=["balanced_accuracy", "roc_auc", "f1_good"], ax=axes[1], title="Validation metrics")
plt.tight_layout()

training_metadata = {
    "model": "PPGQualityResNet1D",
    "checkpoint": str(checkpoint_path.relative_to(REPO_ROOT)),
    "input_shape": [3, 300],
    "view_order": ["raw_robust", "butterworth_bandpass", "first_derivative"],
    "selection_metric": "validation balanced accuracy at threshold 0.5",
    "best_validation_balanced_accuracy": float(best_score),
    "epochs_completed": len(history),
    "seed": SEED,
    "parameter_count": parameter_count(model),
    "class_weight_good": float(positive_weight.item()),
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "waveform_training_metadata.json").write_text(json.dumps(training_metadata, indent=2) + "\n")
training_metadata